# Classificação de Doenças em Plantas com Deep Learning

**Aluno:** Thomaz Ritter

Relatório completo: [`relatorio_desafio.md`](relatorio_desafio.md)

---

Neste notebook, vamos treinar redes neurais convolucionais (CNNs) para classificar doenças em plantas
a partir de fotos de folhas. São 38 classes possíveis (14 espécies, entre saudável ou doença específica).

Vamos comparar **três abordagens** para entender os trade-offs:
1. ResNet18 com augmentation leve (baseline)
2. ResNet50 com augmentation leve (modelo maior melhora?)
3. ResNet18 com augmentation agressiva (mais augmentation melhora?)

## 1. Setup e Instalação

In [ ]:
!pip install -q kagglehub

Importamos o PyTorch (framework de deep learning), torchvision (modelos pré-treinados e transforms),
matplotlib/seaborn (gráficos) e sklearn (métricas de avaliação).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import torchvision.transforms as transforms
from torchvision import datasets
from torchvision.models import resnet18, ResNet18_Weights, resnet50, ResNet50_Weights

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support
import seaborn as sns
from collections import Counter
from PIL import Image
import os

Verificamos qual hardware está disponível. No Colab com GPU, usa CUDA.
No Mac com Apple Silicon, usa MPS. Senão, CPU (vai ser mais lento).

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print('Usando GPU CUDA')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Usando Apple Silicon (MPS)')
else:
    device = torch.device('cpu')
    print('Usando CPU (vai demorar mais)')

### Hiperparâmetros

Esses são os valores que controlam o treinamento. A escolha de cada um está justificada no relatório (seção 3.3).

In [ ]:
BATCH_SIZE = 64       # imagens processadas por vez
EPOCHS = 10           # passadas completas pelo dataset
LR = 1e-4             # learning rate (baixo para nao destruir pesos pre-treinados)
TRAIN_SPLIT = 0.8     # 80% treino, 20% validacao
SEED = 42             # seed fixa para reprodutibilidade
NUM_WORKERS = 2       # threads para carregar dados em paralelo

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Batch: {BATCH_SIZE} | Epochs: {EPOCHS} | LR: {LR}')

## 2. Dataset PlantVillage

O PlantVillage (Hughes & Salathe, 2015) tem ~54 mil imagens de folhas, organizadas em 38 classes.
Vamos baixar direto do Kaggle usando a lib `kagglehub`.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
DATA_DIR = os.path.join(path, 'plantvillage dataset', 'color')

print(f"Dataset em: {DATA_DIR}")
print(f"Classes encontradas: {len(os.listdir(DATA_DIR))}")

In [ ]:
# Listar todas as classes e quantas imagens cada uma tem
CLASSES = sorted(os.listdir(DATA_DIR))
NUM_CLASSES = len(CLASSES)

print(f'Dataset com {NUM_CLASSES} classes:\n')
for i, cls in enumerate(CLASSES):
    n_imgs = len(os.listdir(os.path.join(DATA_DIR, cls)))
    print(f'  {i:2d}. {cls} ({n_imgs} imagens)')

### Pré-processamento

Todas as imagens passam por:
- **Resize** para 224×224 (tamanho que a ResNet espera)
- **Normalização** com média e desvio padrão do ImageNet (fundamental para transfer learning)

No treino, adicionamos um **RandomHorizontalFlip** (espelha a imagem com 50% de chance).
Na validação, nenhum augmentation, queremos avaliar o modelo em condições limpas.

In [ ]:
# Transform de treino com augmentation leve, resize e normalize
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),        # espelha horizontalmente (50% de chance)
    transforms.Resize((224, 224)),             # tamanho padrao da ResNet
    transforms.ToTensor(),                     # converte para tensor [0, 1]
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),            # media do ImageNet
        std=(0.229, 0.224, 0.225)              # desvio padrao do ImageNet
    )
])

# Transform de validacao, sem augmentation
transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

### Dividindo em treino e validação

80% das imagens vão para o treino, 20% para validação. A seed fixa garante que a divisão é sempre a mesma.

In [ ]:
# Carregar todas as imagens como um dataset
full_dataset = datasets.ImageFolder(DATA_DIR, transform=transform_train)

# Dividir 80/20
train_size = int(TRAIN_SPLIT * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Na validacao, usamos transform sem augmentation
val_dataset.dataset = datasets.ImageFolder(DATA_DIR, transform=transform_test)

# DataLoaders para iterar em batches
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'Treino: {len(train_dataset)} imagens | Validacao: {len(val_dataset)} imagens')

### Visualizando o dataset

In [ ]:
# Amostras do dataset, como sao as folhas que o modelo vai ver
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(18, 5))

for i, ax in enumerate(axes.flat):
    # Desfazer a normalizacao para visualizar
    img = images[i].permute(1, 2, 0).numpy()
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])

    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(CLASSES[labels[i]].replace('___', '\n'), fontsize=6)
    ax.axis('off')

plt.suptitle('Amostras do Dataset PlantVillage', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Distribuicao de classes, nem todas tem a mesma quantidade de imagens
class_counts = Counter(full_dataset.targets)
names = [CLASSES[i].replace('___', ' - ') for i in range(NUM_CLASSES)]
counts = [class_counts[i] for i in range(NUM_CLASSES)]

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(names, counts, color='steelblue')
ax.set_xlabel('Numero de Imagens')
ax.set_title('Distribuicao de Classes no PlantVillage')
plt.tight_layout()
plt.show()

## 3. Experimentos

### Função de treinamento

Antes dos experimentos, definimos uma função reutilizável que faz o loop clássico de treinamento, que é
**forward pass → calcula loss → backward pass → atualiza pesos**.

Ela retorna o histórico de acurácia e loss para gerar os gráficos depois.

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs):
    """Treina o modelo e retorna o historico de metricas."""
    history = {'train_acc': [], 'train_loss': [], 'val_acc': [], 'val_loss': []}

    for epoch in range(1, epochs + 1):

        # === FASE DE TREINO ===
        model.train()
        correct, total, running_loss = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()          # zera gradientes da iteracao anterior
            out = model(images)            # forward pass
            loss = criterion(out, labels)  # calcula o erro
            loss.backward()                # backward pass (calcula gradientes)
            optimizer.step()               # atualiza os pesos

            running_loss += loss.item() * images.size(0)
            correct += out.argmax(1).eq(labels).sum().item()
            total += images.size(0)

        train_loss = running_loss / total
        train_acc = 100. * correct / total
        history['train_acc'].append(train_acc)
        history['train_loss'].append(train_loss)

        # === FASE DE VALIDACAO ===
        model.eval()
        correct, total, running_loss = 0, 0, 0

        with torch.no_grad():  # desliga gradientes (nao precisamos na validacao)
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                out = model(images)
                loss = criterion(out, labels)

                running_loss += loss.item() * images.size(0)
                correct += out.argmax(1).eq(labels).sum().item()
                total += images.size(0)

        val_loss = running_loss / total
        val_acc = 100. * correct / total
        history['val_acc'].append(val_acc)
        history['val_loss'].append(val_loss)

        scheduler.step()  # ajusta o learning rate

        print(f'Epoch {epoch}/{epochs} | '
              f'Loss {train_loss:.4f} / {val_loss:.4f} | '
              f'Acc {train_acc:.2f}% / {val_acc:.2f}%')

    return history

In [ ]:
def get_predictions(model, loader):
    """Coleta predicoes e labels reais para gerar metricas detalhadas."""
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            preds = model(images).argmax(1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_labels)

### 3.1 Experimento 1: ResNet18 + Augmentation Leve (Baseline)

Nosso baseline usa a ResNet18 (11.7M parâmetros) com fine-tuning completo.
A única coisa que trocamos é a última camada, de 1000 classes (ImageNet) para 38 (PlantVillage).

In [ ]:
# Carregar ResNet18 pre-treinada no ImageNet
model1 = resnet18(weights=ResNet18_Weights.DEFAULT)

# Trocar a ultima camada para 38 classes
model1.fc = nn.Linear(model1.fc.in_features, NUM_CLASSES)
model1 = model1.to(device)

# Configurar otimizador e scheduler
criterion1 = nn.CrossEntropyLoss()
optimizer1 = optim.Adam(model1.parameters(), lr=LR)
scheduler1 = optim.lr_scheduler.CosineAnnealingLR(optimizer1, T_max=EPOCHS)

print(f'ResNet18 - Parametros: {sum(p.numel() for p in model1.parameters()):,}')

In [ ]:
print('=== Experimento 1: ResNet18 + Augmentation Leve ===')
history1 = train_model(model1, train_loader, val_loader, criterion1, optimizer1, scheduler1, EPOCHS)

### 3.2 Experimento 2: ResNet50 + Augmentation Leve

Mesmo setup do Exp1, mas agora com a ResNet50 (25.6M parâmetros, o dobro).
A pergunta aqui é se mais parâmetros significam melhor resultado.

In [ ]:
model2 = resnet50(weights=ResNet50_Weights.DEFAULT)
model2.fc = nn.Linear(model2.fc.in_features, NUM_CLASSES)
model2 = model2.to(device)

criterion2 = nn.CrossEntropyLoss()
optimizer2 = optim.Adam(model2.parameters(), lr=LR)
scheduler2 = optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=EPOCHS)

print(f'ResNet50 - Parametros: {sum(p.numel() for p in model2.parameters()):,}')

In [ ]:
print('=== Experimento 2: ResNet50 + Augmentation Leve ===')
history2 = train_model(model2, train_loader, val_loader, criterion2, optimizer2, scheduler2, EPOCHS)

### 3.3 Experimento 3: ResNet18 + Augmentation Agressiva

Mesmo modelo do Exp1 (ResNet18), mas com augmentation muito mais forte.
A ideia é simular condições do mundo real, como folhas tortas, com sombra, cores variando.

Comparamos visualmente as duas abordagens abaixo.

In [ ]:
# Transforms com augmentation agressiva
transform_aggressive = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),                                          # gira ate 30 graus
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),  # varia cores
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),     # desloca e escala
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

# Visualizar a diferenca entre augmentation leve e agressiva na mesma imagem
fig, axes = plt.subplots(2, 6, figsize=(15, 5))

sample_img = full_dataset.samples[0][0]
original = Image.open(sample_img)

for i in range(6):
    # Leve
    img_leve = transform_train(original).permute(1, 2, 0).numpy()
    img_leve = img_leve * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    axes[0, i].imshow(np.clip(img_leve, 0, 1))
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Leve', fontsize=12)

    # Agressiva
    img_agr = transform_aggressive(original).permute(1, 2, 0).numpy()
    img_agr = img_agr * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    axes[1, i].imshow(np.clip(img_agr, 0, 1))
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Agressiva', fontsize=12)

plt.suptitle('Comparacao: Data Augmentation Leve vs Agressiva', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Recriar dataset com augmentation agressiva
aug_dataset = datasets.ImageFolder(DATA_DIR, transform=transform_aggressive)

aug_train_dataset, aug_val_dataset = random_split(
    aug_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Validacao sempre sem augmentation, avaliamos em condicoes limpas
aug_val_dataset.dataset = datasets.ImageFolder(DATA_DIR, transform=transform_test)

aug_train_loader = DataLoader(aug_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
aug_val_loader = DataLoader(aug_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
model3 = resnet18(weights=ResNet18_Weights.DEFAULT)
model3.fc = nn.Linear(model3.fc.in_features, NUM_CLASSES)
model3 = model3.to(device)

criterion3 = nn.CrossEntropyLoss()
optimizer3 = optim.Adam(model3.parameters(), lr=LR)
scheduler3 = optim.lr_scheduler.CosineAnnealingLR(optimizer3, T_max=EPOCHS)

In [ ]:
print('=== Experimento 3: ResNet18 + Augmentation Agressiva ===')
history3 = train_model(model3, aug_train_loader, aug_val_loader, criterion3, optimizer3, scheduler3, EPOCHS)

## 4. Resultados e Métricas

### 4.1 Curvas de aprendizado

Vamos comparar os três experimentos lado a lado, comparando acurácia de validação e loss ao longo das épocas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
epochs_range = range(1, EPOCHS + 1)

# Acuracia
axes[0].plot(epochs_range, history1['val_acc'], 'o-', label='Exp1: ResNet18 + Aug. Leve', color='#2196F3')
axes[0].plot(epochs_range, history2['val_acc'], 's-', label='Exp2: ResNet50 + Aug. Leve', color='#FF5722')
axes[0].plot(epochs_range, history3['val_acc'], '^-', label='Exp3: ResNet18 + Aug. Agressiva', color='#4CAF50')
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Acuracia (%)')
axes[0].set_title('Acuracia de Validacao')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(epochs_range, history1['val_loss'], 'o-', label='Exp1: ResNet18 + Aug. Leve', color='#2196F3')
axes[1].plot(epochs_range, history2['val_loss'], 's-', label='Exp2: ResNet50 + Aug. Leve', color='#FF5722')
axes[1].plot(epochs_range, history3['val_loss'], '^-', label='Exp3: ResNet18 + Aug. Agressiva', color='#4CAF50')
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss de Validacao')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Comparacao dos Tres Experimentos', fontsize=14)
plt.tight_layout()
plt.show()

### 4.2 Treino vs Validação (overfitting?)

Se a acurácia de treino for muito maior que a de validação, o modelo está memorizando em vez de aprender.
Vamos checar isso para cada experimento.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, hist, titulo in zip(axes,
    [history1, history2, history3],
    ['Exp1: ResNet18 + Aug. Leve', 'Exp2: ResNet50 + Aug. Leve', 'Exp3: ResNet18 + Aug. Agressiva']):

    ax.plot(epochs_range, hist['train_acc'], 'o-', label='Treino', color='#2196F3')
    ax.plot(epochs_range, hist['val_acc'], 's-', label='Validacao', color='#FF5722')
    ax.set_xlabel('Epoca')
    ax.set_ylabel('Acuracia (%)')
    ax.set_title(titulo)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Treino vs Validacao (Analise de Overfitting)', fontsize=14)
plt.tight_layout()
plt.show()

### 4.3 Tabela comparativa

In [ ]:
print('=' * 80)
print(f'{"Experimento":<45} {"Acc Treino":>12} {"Acc Valid":>12} {"Loss Valid":>12}')
print('=' * 80)

for nome, hist in [('Exp1: ResNet18 + Aug. Leve', history1),
                    ('Exp2: ResNet50 + Aug. Leve', history2),
                    ('Exp3: ResNet18 + Aug. Agressiva', history3)]:
    print(f'{nome:<45} {hist["train_acc"][-1]:>11.2f}% {hist["val_acc"][-1]:>11.2f}% {hist["val_loss"][-1]:>12.4f}')

print('=' * 80)

### 4.4 Melhor modelo

In [ ]:
# Qual modelo teve a melhor acuracia de validacao?
best_val_accs = {
    'Exp1: ResNet18 + Aug. Leve': max(history1['val_acc']),
    'Exp2: ResNet50 + Aug. Leve': max(history2['val_acc']),
    'Exp3: ResNet18 + Aug. Agressiva': max(history3['val_acc'])
}

melhor = max(best_val_accs, key=best_val_accs.get)
print(f'Melhor modelo: {melhor} com {best_val_accs[melhor]:.2f}% de acuracia')

# Selecionar modelo e loader para analise detalhada
best_model = {'Exp1: ResNet18 + Aug. Leve': model1,
              'Exp2: ResNet50 + Aug. Leve': model2,
              'Exp3: ResNet18 + Aug. Agressiva': model3}[melhor]

best_val_loader = {'Exp1: ResNet18 + Aug. Leve': val_loader,
                   'Exp2: ResNet50 + Aug. Leve': val_loader,
                   'Exp3: ResNet18 + Aug. Agressiva': aug_val_loader}[melhor]

### 4.5 Matriz de confusão

A matriz de confusão mostra, para cada classe real (eixo Y), o que o modelo previu (eixo X).
A diagonal principal são os acertos, quanto mais azul escuro, melhor.

In [ ]:
preds, labels_true = get_predictions(best_model, best_val_loader)

cm = confusion_matrix(labels_true, preds)

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=[c.replace('___', '\n') for c in CLASSES],
            yticklabels=[c.replace('___', '\n') for c in CLASSES])
ax.set_xlabel('Predicao')
ax.set_ylabel('Real')
ax.set_title(f'Matriz de Confusao - {melhor}')
plt.xticks(rotation=90, fontsize=6)
plt.yticks(fontsize=6)
plt.tight_layout()
plt.show()

### 4.6 Classification report

Precision, recall e F1-score por classe. Essas métricas são mais informativas que acurácia
quando o dataset é desbalanceado.

In [ ]:
report = classification_report(
    labels_true, preds,
    target_names=[c.replace('___', ' - ') for c in CLASSES],
    digits=3
)
print(f'Classification Report - {melhor}\n')
print(report)

### 4.7 Classes com pior desempenho

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    labels_true, preds, average=None
)

worst_idx = np.argsort(f1)[:5]

print('Top 5 classes com PIOR F1-Score:\n')
print(f'{"Classe":<55} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 85)
for idx in worst_idx:
    nome = CLASSES[idx].replace('___', ' - ')
    print(f'{nome:<55} {precision[idx]:>10.3f} {recall[idx]:>10.3f} {f1[idx]:>10.3f}')

### 4.8 Exemplos de acertos e erros

Visualização de imagens que o modelo acertou (verde) e errou (vermelho).
Nos erros, mostramos a classe real e a classe prevista.

In [ ]:
# Recarregar dataset sem shuffle para pegar imagens especificas
val_display_dataset = datasets.ImageFolder(DATA_DIR, transform=transform_test)
_, val_display = random_split(
    val_display_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

erros_idx = np.where(preds != labels_true)[0]
acertos_idx = np.where(preds == labels_true)[0]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

# Acertos
for i in range(5):
    idx = acertos_idx[i]
    img, _ = val_display[idx]
    img_np = img.permute(1, 2, 0).numpy()
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])

    axes[0, i].imshow(np.clip(img_np, 0, 1))
    axes[0, i].set_title(f'Pred: {CLASSES[preds[idx]][:20]}', fontsize=8, color='green')
    axes[0, i].axis('off')

axes[0, 0].set_ylabel('ACERTOS', fontsize=12, color='green')

# Erros
for i in range(min(5, len(erros_idx))):
    idx = erros_idx[i]
    img, _ = val_display[idx]
    img_np = img.permute(1, 2, 0).numpy()
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])

    axes[1, i].imshow(np.clip(img_np, 0, 1))
    axes[1, i].set_title(f'Real: {CLASSES[labels_true[idx]][:15]}\nPred: {CLASSES[preds[idx]][:15]}',
                         fontsize=7, color='red')
    axes[1, i].axis('off')

axes[1, 0].set_ylabel('ERROS', fontsize=12, color='red')

plt.suptitle('Exemplos de Predicoes', fontsize=14)
plt.tight_layout()
plt.show()